# Exercise (Fashion MNIST) - 60 minutes
## Opis zbioru danych
Zadanie dotyczy rozpoznawania kształtów w zbiorze Fashion MNIST. Zbiór ten składa się z 70 000 (60 000 w zbiorze uczącym i 10 000 w zbiorze testowym) czarno-białych obrazków należących do 10 kategorii. Każdy obrazek przedstawia jedno z 10 ubrań (z asortymentu Zalando) w stosunkowo słabej rozdzielczości 28x28 pikseli. Na poniższym rysunku przedstawionych jest 25 pierwszych obrazków ze zbioru uczącego.

Każdy piksel ma skojarzoną z nim liczbę, która określa natężenie koloru (większa wartość oznacza ciemniejszy piksel). Zmienna ta przyjmuje wartości od 0 do 255. Zbiór uczący ma 785 kolumn ($28 \times 28 = 784$ piksele oraz etykietę klasy). Więcej informacji o zbiorze danych można znaleźć na stronie [Kaggle](https://www.kaggle.com/zalando-research/fashionmnist). Zbiór można załadować używając funkcji `dataset_fashion_mnist()` z biblioteki `keras`.

Państwa zadaniem jest skonstruowanie głębokiej sieci neuronowej (minimum trzy warstwy), która będzie klasyfikowała elementy zbioru testowego. Sieć ma być nauczona jedynie na zbiorze uczącym. Miarą jakości rozwiazania jest procent poprawnej klasyfikacji, który powinien przekroczyć 87% na biorze testowym. Najlepsze obecnie rozwiązania dla tego zbioru danych osiągają ponad 97% skuteczność.

## Wymagania

1. Wyuczenie modelu (minimum 50 epok) na zbiorze uczącym.
2. Wizualne przedstawienie procesu uczenia.
3. Pokazanie skuteczności modelu na zbiorze testowym.
4. Przestawienie wyników w postaci macierzy przynależności na zbiorze testowym.
5. Porównanie rozwiązania DNN z lasem losowym (RF) oraz XGBoost.

In [ ]:
# Colab installs (shell)
!pip install xgboost
!pip install tensorflow

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import random

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:

# Load the Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Normalize the images
train_images = train_images / 255.0
test_images = test_images / 255.0

# Class names for Fashion MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Plot the first 25 images from the training set with their class names
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_labels[i]])
plt.show()

## Trenowanie modeli i porownanie (DNN vs RF vs XGBoost)

W tej czesci:
1. Trenujemy siec DNN (minimum 50 epok).
2. Wizualizujemy przebieg uczenia.
3. Oceniamy skutecznosc na zbiorze testowym.
4. Tworzymy macierz pomylek.
5. Porownujemy wynik DNN z Random Forest i XGBoost.

In [ ]:
# Reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# DNN: dane wejsciowe 28x28 zostana splaszczone przez warstwe Flatten.
# RF/XGBoost: potrzebne dane 2D (n_samples, 784)
X_train_flat = train_images.reshape(len(train_images), -1)
X_test_flat = test_images.reshape(len(test_images), -1)

print("train_images shape:", train_images.shape)
print("test_images shape:", test_images.shape)
print("X_train_flat shape:", X_train_flat.shape)
print("X_test_flat shape:", X_test_flat.shape)

## 1) Deep Neural Network (DNN)
Model ma co najmniej 3 warstwy uczace i jest trenowany przez 50 epok.

In [ ]:
dnn_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

dnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = dnn_model.fit(
    train_images, train_labels,
    epochs=50,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('DNN - Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.title('DNN - Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = dnn_model.evaluate(test_images, test_labels, verbose=0)
print(f"DNN test accuracy: {test_acc:.4f}")

dnn_probs = dnn_model.predict(test_images, verbose=0)
dnn_pred = np.argmax(dnn_probs, axis=1)

print("\nClassification report (DNN):")
print(classification_report(test_labels, dnn_pred, target_names=class_names))

cm_dnn = confusion_matrix(test_labels, dnn_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_dnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - DNN')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 2) Porownanie z Random Forest i XGBoost

Modele RF i XGBoost trenujemy na tym samym zbiorze uczacym co DNN oraz oceniamy na tym samym zbiorze testowym.

In [ ]:
# Trening klasycznych modeli na tym samym zbiorze uczacym co DNN
# (pelny train) oraz ocena na tym samym zbiorze testowym (pelny test)

rf_model = RandomForestClassifier(
    n_estimators=80,
    max_depth=14,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_model.fit(X_train_flat, train_labels)
rf_pred = rf_model.predict(X_test_flat)
rf_acc = accuracy_score(test_labels, rf_pred)
print(f"RF test accuracy: {rf_acc:.4f}")

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=80,
    max_depth=4,
    learning_rate=0.2,
    subsample=0.7,
    colsample_bytree=0.7,
    tree_method='hist',
    objective='multi:softmax',
    num_class=10,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_flat, train_labels, verbose=True)
xgb_pred = xgb_model.predict(X_test_flat)
xgb_acc = accuracy_score(test_labels, xgb_pred)
print(f"XGBoost test accuracy: {xgb_acc:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": ["DNN", "Random Forest", "XGBoost"],
    "Test Accuracy": [test_acc, rf_acc, xgb_acc]
}).sort_values("Test Accuracy", ascending=False)

results

## Wnioski

- Model DNN został wytrenowany przez 50 epok, osiągając na zbiorze testowym dokładność **89.60%**, co spełnia wymaganie przekroczenia 87% celności.

- **Wizualizacja procesu uczenia:** Wykresy straty (loss) i dokładności (accuracy) pokazują, że model skutecznie uczył się na zbiorze treningowym. Dokładność na zbiorze walidacyjnym rosła stabilnie przez większość epok, a strata malała, wskazując na dobrą konwergencję. Po około 40 epokach można zaobserwować delikatną stabilizację lub minimalne pogorszenie na zbiorze walidacyjnym, co może sugerować niewielkie przetrenowanie.

- **Skuteczność na zbiorze testowym:** Model osiągnął na zbiorze testowym dokładność **89.60%**.

- **Analiza macierzy pomyłek dla DNN:** Macierz pomyłek ujawnia, że model najlepiej radzi sobie z klasyfikacją `Trouser` , `Sandal`  i `Bag` , dla których osiąga bardzo wysoką poprawność klasyfikacji. Największe trudności występują przy rozróżnianiu wizualnie podobnych kategorii, takich jak `Shirt` , która jest często mylona z `T-shirt/top` oraz `Pullover` i `Coat` , które również są ze sobą wzajemnie mylone. Klasa `Shirt` charakteryzuje się najniższą precyzją i czułością wśród wszystkich kategorii.

- **Porównanie z innymi klasyfikatorami:** W porównaniu do innych klasyfikatorów, DNN (89.60%) osiągnął najwyższą dokładność, przewyższając XGBoost (88.09%) i Random Forest (86.65%).